# Hybrid RAG: a hands-on course

*Lecture notebook. Read the markdown like lecture notes, run the code like a lab.*

## Learning objectives
By the end you should be able to:
1. Explain **why** neither keyword search nor vector search is enough on its own.
2. Implement **BM25** from scratch and say what each knob (`k1`, `b`) does.
3. Explain how **embeddings** turn meaning into geometry, and what cosine similarity measures.
4. Fuse two rankings with **Reciprocal Rank Fusion (RRF)** and with a **weighted blend**, and choose between them.
5. **Measure** a retriever with recall@k, MRR and nDCG, and read per-query failures.
6. Explain what a **cross-encoder reranker** adds and what it costs.

## The big picture
```
                 user query
                     |
        +------------+------------+
        |                         |
   BM25 (sparse)           Embeddings (dense)
   exact words             meaning
        |                         |
   ranked list A             ranked list B
        +------------+------------+
                     |
              FUSION (RRF / weighted)
                     |
              top-N candidates
                     |
          CROSS-ENCODER RERANKER   (optional, slow, precise)
                     |
                top-k chunks  --->  LLM prompt  --->  answer
```
This notebook builds everything up to the LLM prompt. Retrieval quality is the ceiling for RAG quality:
if the right chunk is not retrieved, no model can answer from it.

## How to use this notebook
- Run cells top to bottom the first time.
- Every section has: **Concept** (the lecture), **Code** (the lab), **Checkpoint** (questions to answer before moving on).
- The last sections hold a **glossary**, **cheat sheet** and **reading list** for later reference.
- **Slide deck** to go with this notebook: https://claude.ai/artifact/1oVsF7vHabo4C9K4PNoVpB (private link, open it while signed in to Claude)
- Companion files in this folder: `hybrid_rag_notes.md` (summary), `hybrid_retriever.py` (a compact script version).

In [ ]:
# Lab setup. If you selected the project's .venv kernel, these packages are already installed
# and this cell does nothing. Safe to skip.
%pip install -q rank-bm25 sentence-transformers numpy pandas matplotlib

## 0. Why hybrid? (the motivating problem)

Two classic ways to find text, and each one fails in a predictable way.

| | **Keyword / sparse (BM25)** | **Semantic / dense (embeddings)** |
|---|---|---|
| Matches on | the *exact tokens* in the query | the *meaning* of the query |
| Great at | IDs, SKUs, error codes, names, rare jargon | paraphrases, synonyms, vague questions |
| Fails at | "forgot password" vs "lost credentials" (no shared words) | `E4021` vs `E4012` (looks similar, means different) |
| Cost | cheap, no model | needs an embedding model and a vector index |
| Explainable? | yes (you can see the matching terms) | no (a 384-number vector) |

**Key insight:** their failures are *different*. Where one is blind, the other can see. Combining them
gives better recall than either alone. That is the whole idea of hybrid retrieval.

**Professor's note:** hybrid is not magic. On some datasets one method already wins and hybrid adds nothing.
That is why section 5 (evaluation) matters: you measure instead of assume.

## 1. The lab corpus and labeled queries

**Concept.** To improve a retriever you must be able to *measure* it. Measuring needs a **test set**:
queries paired with the documents that are truly relevant (called **qrels**, short for "query relevance judgments").

Our corpus is small on purpose (18 docs) so you can read every document and understand every result.
Queries come in two flavors:
- **Exact-token** queries (`E4021`, `AX-9920`): BM25 should shine.
- **Paraphrase** queries ("I forgot my password"): dense should shine.

**Warning:** with 18 documents this is a *toy*. Real systems have thousands to millions of chunks, and
differences between methods show up much more clearly there (see Exercise 8).

In [ ]:
# DOCS: our knowledge base. Key = document id, value = the text of one "chunk".
# In a real RAG system each entry would be a chunk of a longer document (a few hundred tokens).
DOCS = {
    0: "Error E4021: connection pool exhausted. Increase max_connections in the database config.",
    1: "To reset your password, click 'Forgot password' on the login page and follow the emailed link.",
    2: "SKU AX-9920 is a stainless steel water bottle with a 750ml capacity.",
    3: "Refunds are available within 30 days of purchase if the item is unused.",
    4: "If you cannot sign in because you lost your credentials, use the account recovery flow.",
    5: "The server rejected new clients because too many sessions were open at once.",
    6: "Standard shipping takes 3 to 5 business days within the country.",
    7: "Restart a Kubernetes deployment with kubectl rollout restart deployment/<name>.",
    8: "SKU BZ-1104 is an insulated lunch box with two compartments.",
    9: "International orders may take 10 to 14 days and can incur customs fees.",
    10: "Error E5090: request timed out while waiting for the upstream payment service.",
    11: "Two-factor authentication can be enabled under Settings > Security.",
    12: "To return an item, print the prepaid label from your orders page and drop the parcel at any carrier.",
    13: "Pods stuck in CrashLoopBackOff usually mean the container exits immediately; check logs with kubectl logs.",
    14: "Our warranty covers manufacturing defects for one year from the delivery date.",
    15: "The payment gateway did not respond in time, so the transaction was cancelled.",
    16: "Gift cards never expire and can be combined with discount codes at checkout.",
    17: "Use the --previous flag with kubectl logs to read logs from a crashed container.",
}

# Two parallel lists so that "position i" in any array corresponds to document DOC_IDS[i].
# Many bugs in retrieval code come from mixing up "position" and "document id". Keep them straight.
DOC_IDS = list(DOCS.keys())
DOC_TEXTS = [DOCS[i] for i in DOC_IDS]

# QRELS: the ground truth. For each query, the SET of document ids a human judged relevant.
# Note that several queries have 2 relevant docs: recall@k needs this to be meaningful.
QRELS = {
    # exact-token style (lexical match should work)
    "E4021": {0},
    "AX-9920": {2},
    "E5090": {10},
    "restart kubernetes deployment": {7},
    "enable 2FA": {11},
    # paraphrase style (needs semantics)
    "I forgot my password": {1, 4},
    "cannot log in lost credentials": {4, 1},
    "database refuses connections": {0, 5},
    "how long does delivery take": {6, 9},
    "send back a purchase for money back": {3, 12},
    "payment timed out": {10, 15},
    "pod keeps crashing": {13, 17},
    "how to get logs from a crashed pod": {17, 13},
    "defective product coverage": {14},
}
print(len(DOCS), "docs,", len(QRELS), "queries")

**Checkpoint 1.** Pick three queries and predict *before running anything* which method (BM25 or dense) will rank the relevant doc first, and why.
Write your predictions down. You will check them in section 6.

## 2. BM25: keyword search done right

**Concept.** BM25 (Best Match 25) is the workhorse behind Lucene, Elasticsearch, OpenSearch and most "classic" search.
It improves TF-IDF with two ideas.

For query `q` and document `d`:

$$\text{score}(d,q)=\sum_{t\in q}\text{IDF}(t)\cdot\frac{tf(t,d)\,(k_1+1)}{tf(t,d)+k_1\left(1-b+b\,\frac{|d|}{\text{avgdl}}\right)}$$

Read it piece by piece:

| Piece | Intuition |
|---|---|
| `tf(t,d)` | how many times term `t` appears in document `d` |
| `IDF(t)` | **rare words matter more.** "the" appears everywhere (low IDF); `e4021` appears once (high IDF) |
| `k1` | **term-frequency saturation.** The 10th occurrence of a word adds much less than the 1st. Larger `k1` = saturates later |
| `b` | **length normalization.** A match in a short doc is stronger evidence than the same match in a long doc. `b=0` ignores length, `b=1` fully normalizes |
| `avgdl` | average document length, the reference point for "long" vs "short" |

BM25 is *sparse*: documents are huge vectors over the whole vocabulary, almost all zeros. Fast with an inverted index.

**Limits:** it only sees tokens. No synonyms, no word order, no meaning.

In [ ]:
import math, re
from collections import Counter
import numpy as np


def tokenize(text):
    # Tokenization decides what BM25 can "see". This one lowercases and keeps identifiers whole:
    # "AX-9920" stays one token ("ax-9920"), and "E4021" becomes "e4021".
    # A classic bug is a tokenizer that splits "AX-9920" into "ax" and "9920", which destroys exact-ID search.
    return re.findall(r"[a-z0-9][a-z0-9_\-]*", text.lower())


class MyBM25:
    """BM25 written out in full so every term of the formula is visible."""

    def __init__(self, corpus_tokens, k1=1.5, b=0.75):
        self.k1, self.b = k1, b                      # k1: tf saturation, b: length normalization

        # Term frequencies per document: tf[i][term] = count of term in doc i.
        self.tf = [Counter(doc) for doc in corpus_tokens]

        # Document lengths (in tokens) and their average.
        self.dl = np.array([len(doc) for doc in corpus_tokens])
        self.avgdl = self.dl.mean()

        # Document frequency: in how many documents does each term appear at least once?
        n = len(corpus_tokens)
        df = Counter(t for doc in corpus_tokens for t in set(doc))

        # IDF (Lucene-style, always positive). Rare term -> large IDF, common term -> small IDF.
        self.idf = {t: math.log(1 + (n - d + 0.5) / (d + 0.5)) for t, d in df.items()}

    def scores(self, query_tokens):
        out = np.zeros(len(self.tf))                  # one score per document, initially 0
        for i, tf in enumerate(self.tf):              # loop over documents
            for t in query_tokens:                    # loop over query terms
                if t not in tf:
                    continue                          # term absent from doc: contributes nothing
                # Numerator: raw tf, boosted by (k1+1) so a single hit has weight about 1.
                num = tf[t] * (self.k1 + 1)
                # Denominator: saturates tf (the +tf) and penalizes long documents (the dl/avgdl part).
                den = tf[t] + self.k1 * (1 - self.b + self.b * self.dl[i] / self.avgdl)
                out[i] += self.idf[t] * num / den     # IDF weights how informative the term is
        return out


corpus_tokens = [tokenize(t) for t in DOC_TEXTS]     # tokenize every document once, up front
my_bm25 = MyBM25(corpus_tokens)

q = "E4021"
s = my_bm25.scores(tokenize(q))
print("top doc for", q, "->", DOC_IDS[int(np.argmax(s))], DOC_TEXTS[int(np.argmax(s))])

In [ ]:
# Sanity check against a well-known library. Good habit: validate your own implementation against a reference.
# Scores differ slightly (the library uses a different IDF variant), but rankings should be very close.
from rank_bm25 import BM25Okapi

lib_bm25 = BM25Okapi(corpus_tokens, k1=1.5, b=0.75)   # same k1/b as ours, otherwise the comparison is unfair
for q in ["E4021", "I forgot my password", "payment timed out"]:
    # np.argsort sorts ascending, so we negate the scores to get "best first". [:3] keeps the top 3.
    mine = np.argsort(-my_bm25.scores(tokenize(q)))[:3]
    lib = np.argsort(-lib_bm25.get_scores(tokenize(q)))[:3]
    # int(...) converts numpy integers so the printout is readable.
    print(f"{q!r:30} mine={[int(i) for i in mine]} lib={[int(i) for i in lib]}")

**How to read that output.** Each number is a document id. For `E4021` only doc 0 contains the token, so only position 1 is meaningful:
every other doc scores exactly 0, and `argsort` returns tied zeros in arbitrary order.
**Rule of thumb:** a ranking is only informative as far down as the scores are non-zero.

### Visual: what `k1` and `b` actually do
Run the next cell and look at both plots. They are the single best way to build intuition for BM25's two knobs.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))

# Left plot: TERM-FREQUENCY SATURATION. Weight contributed by a term as its count in the doc grows.
tf = np.arange(0, 21)
for k1 in [0.5, 1.2, 2.0, 5.0]:
    axes[0].plot(tf, tf * (k1 + 1) / (tf + k1), label=f"k1={k1}")
axes[0].plot(tf, tf, "k:", label="raw tf (no saturation)")
axes[0].set_ylim(0, 7)
axes[0].set_xlabel("term count in document (tf)")
axes[0].set_ylabel("tf component of the score")
axes[0].set_title("Saturation: repeating a word has diminishing returns")
axes[0].legend(fontsize=8)

# Right plot: LENGTH NORMALIZATION. Same tf=3 in documents of different lengths (ratio = dl / avgdl).
ratio = np.linspace(0.25, 4, 100)
k1 = 1.5
for b in [0.0, 0.5, 0.75, 1.0]:
    axes[1].plot(ratio, 3 * (k1 + 1) / (3 + k1 * (1 - b + b * ratio)), label=f"b={b}")
axes[1].set_xlabel("document length / average length")
axes[1].set_ylabel("tf component (tf=3)")
axes[1].set_title("Length norm: long docs are penalized when b is large")
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()

**Reading the plots.**
- Left: every curve flattens. With `k1=0.5` it flattens almost immediately (a word repeated 3 times is nearly as good as 20 times); with `k1=5` it keeps climbing.
- Right: at `b=0` the line is flat (length is ignored). At `b=1` a document 4x longer than average gets a much weaker score for the same `tf`.
- Defaults `k1` about 1.2 to 2 and `b=0.75` work well on most text. Tune them last, not first.

**Checkpoint 2.**
1. Why does BM25 score "I forgot my password" poorly against doc 4 ("lost your credentials")?
2. If your documents are all about the same length, what happens if you set `b=0`?
3. Why is `IDF` essential? What would happen to the query "the database" without it?

## 3. Dense retrieval: meaning as geometry

**Concept.** An **embedding model** (here `all-MiniLM-L6-v2`, a small transformer) maps text to a vector of 384 numbers.
Texts with similar *meaning* land near each other in this 384-dimensional space, even with no shared words.

**Cosine similarity** measures the angle between two vectors:

$$\cos(\theta)=\frac{u\cdot v}{\lVert u\rVert\,\lVert v\rVert}$$

If we **normalize** every vector to length 1, cosine similarity is just the dot product. That is why the code
uses `normalize_embeddings=True` and then `doc_vecs @ query_vec`.

At scale, an **ANN** (approximate nearest neighbor) index such as **HNSW** avoids comparing the query to every vector.
We use brute force here because 18 vectors is nothing.

**Limits:** dense models can blur near-identical identifiers (`E4021` vs `E4012`), struggle with rare words they never saw
in training, and give you no explanation of *why* something matched.

In [ ]:
from sentence_transformers import SentenceTransformer

# Load a pretrained embedding model (downloads ~90MB the first time).
model = SentenceTransformer("all-MiniLM-L6-v2")

# Encode ALL documents once, ahead of time. This is the "indexing" step of dense retrieval.
# normalize_embeddings=True makes each vector unit length, so cosine similarity == dot product.
doc_vecs = model.encode(DOC_TEXTS, normalize_embeddings=True)
print("document matrix shape:", doc_vecs.shape, "(num_docs, embedding_dim)")


# ---- Three tiny helper functions. Every retriever in this notebook is built from these. ----

def dense_scores(query):
    # Encode the query with the SAME model, then one matrix-vector product scores every document at once.
    qv = model.encode([query], normalize_embeddings=True)[0]
    return doc_vecs @ qv                       # shape: (num_docs,), higher = more similar


def bm25_scores(query):
    return my_bm25.scores(tokenize(query))      # shape: (num_docs,), higher = better keyword match


def rank_from_scores(scores):
    # Turn a score array into a list of DOCUMENT IDS ordered best-first.
    # argsort gives positions; we map positions back to ids via DOC_IDS.
    return [DOC_IDS[i] for i in np.argsort(-scores)]


def show(query, ranking, n=3):
    print(f"Query: {query}")
    for d in ranking[:n]:
        print(f"  [{d}] {DOCS[d]}")


for q in ["E4021", "I forgot my password"]:
    print("--- BM25");  show(q, rank_from_scores(bm25_scores(q)))
    print("--- Dense"); show(q, rank_from_scores(dense_scores(q)))

### Visual: the embedding space
We cannot draw 384 dimensions, so we project to 2D with **PCA** (principal component analysis, done here with an SVD).
It is a rough picture, but you should see related documents (passwords, kubernetes, payments, shipping) cluster together,
and a query landing near its meaning.

In [ ]:
# PCA by hand: center the data, take the top-2 right singular vectors as axes.
mean = doc_vecs.mean(axis=0)
_, _, Vt = np.linalg.svd(doc_vecs - mean, full_matrices=False)
proj = lambda X: (X - mean) @ Vt[:2].T          # project any vectors into the same 2D plane

coords = proj(doc_vecs)
fig, ax = plt.subplots(figsize=(7.5, 5.5))
ax.scatter(coords[:, 0], coords[:, 1], s=60, c="tab:blue", alpha=0.7)
for i, (x, y) in zip(DOC_IDS, coords):
    ax.annotate(str(i), (x, y), textcoords="offset points", xytext=(5, 4), fontsize=9)

# Overlay two queries as red stars to see where they land relative to the documents.
for q in ["I forgot my password", "pod keeps crashing"]:
    qx, qy = proj(model.encode([q], normalize_embeddings=True))[0]
    ax.scatter(qx, qy, marker="*", s=260, c="tab:red")
    ax.annotate(q, (qx, qy), textcoords="offset points", xytext=(8, -12), fontsize=9, color="tab:red")

ax.set_title("Documents (blue, numbered by id) and queries (red stars) in 2D PCA")
ax.set_xlabel("PC1"); ax.set_ylabel("PC2")
plt.show()

**Checkpoint 3.**
1. Find docs 13 and 17 (both kubectl logs). Are they near each other? Where is doc 7 (restart deployment)?
2. The 2D picture loses a lot of information. Why might two docs look far apart here but be close in 384 dimensions?

## 4. Fusion: merging two ranked lists

We now have two rankings per query. How do we make one?

**The obstacle: the scores are not comparable.** BM25 scores can be 0 to 15+; cosine scores live between -1 and 1.
Adding them directly lets whichever has the bigger numbers dominate. See it for yourself below.

In [ ]:
q = "payment timed out"
bm = bm25_scores(q)
de = dense_scores(q)

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5), sharex=True)
axes[0].bar(DOC_IDS, bm, color="tab:orange");  axes[0].set_title(f"BM25 raw scores (max={bm.max():.2f})")
axes[1].bar(DOC_IDS, de, color="tab:blue");    axes[1].set_title(f"Dense cosine scores (max={de.max():.2f})")
for ax in axes:
    ax.set_xlabel("document id"); ax.set_xticks(DOC_IDS)
plt.suptitle(f"Query: {q!r}. Different scales, different shapes")
plt.tight_layout(); plt.show()

Notice: BM25 is **spiky** (a few docs with large scores, the rest exactly 0). Dense is **smooth** (every doc has some similarity).
Two common cures:

### Option A: Reciprocal Rank Fusion (RRF)
Ignore scores entirely. Use only each document's **rank** in each list:

$$\text{RRF}(d)=\sum_{r\in\text{rankers}}\frac{1}{k+\text{rank}_r(d)}$$

- `k` (usually **60**) softens the gap between top ranks. Small `k` = the #1 spot is hugely valuable; large `k` = ranks 1 and 5 count almost the same.
- A doc that is *good in both lists* beats a doc that is *great in one and absent in the other*.
- No score normalization, no tuning needed to get a good default. That is why RRF is the standard first choice
  (Elasticsearch, Weaviate, Azure AI Search, Chroma and others all ship it).

### Option B: Weighted blend of normalized scores
First **min-max normalize** each score list to [0, 1], then mix:

$$\text{score}(d)=\alpha\cdot\text{dense}_{norm}(d)+(1-\alpha)\cdot\text{bm25}_{norm}(d)$$

- Uses the *magnitude* of the scores, not just rank, so it can exploit "this match was overwhelming".
- Needs a good `alpha` (tune on your eval set) and is sensitive to outliers in min-max.

**Rule of thumb:** start with RRF. Move to a tuned weighted blend only when you have an eval set that proves it helps.

In [ ]:
def rrf(rankings, k=60):
    # rankings: a list of rankings, each a list of doc ids ordered best-first.
    fused = {}
    for ranking in rankings:
        # enumerate(..., start=1) because RRF uses 1-based ranks (the best doc has rank 1, not 0).
        for rank, d in enumerate(ranking, start=1):
            # Each list "votes" for a doc with weight 1/(k+rank). Votes from different lists add up.
            fused[d] = fused.get(d, 0.0) + 1.0 / (k + rank)
    # Sort docs by total vote, highest first, and return just the ids.
    return [d for d, _ in sorted(fused.items(), key=lambda kv: -kv[1])]


def minmax(x):
    # Rescale scores to [0, 1] so BM25 and cosine become comparable. The guard avoids dividing by zero
    # when every score is identical (e.g. a query that matches nothing in BM25).
    lo, hi = x.min(), x.max()
    return (x - lo) / (hi - lo) if hi > lo else np.zeros_like(x)


def weighted(query, alpha=0.5):
    # alpha=1 -> dense only, alpha=0 -> BM25 only.
    s = alpha * minmax(dense_scores(query)) + (1 - alpha) * minmax(bm25_scores(query))
    return rank_from_scores(s)


def hybrid_rrf(query, k=60, pool=50):
    # pool: how many candidates to take from EACH retriever before fusing (top-N per retriever).
    # In production this is typically 50 to 500. With 18 docs it simply means "everything".
    sparse = rank_from_scores(bm25_scores(query))[:pool]
    dense = rank_from_scores(dense_scores(query))[:pool]
    return rrf([sparse, dense], k=k)


for q in ["E4021", "I forgot my password", "payment timed out"]:
    print("--- Hybrid RRF"); show(q, hybrid_rrf(q))

### Worked example: RRF by hand
Take two toy rankings. Doc `A` is #1 in BM25 and #3 in dense; doc `B` is #2 in both; doc `C` is #1 in dense only.

In [ ]:
import pandas as pd

bm_rank = {"A": 1, "B": 2}                 # C is absent from the BM25 list
de_rank = {"C": 1, "B": 2, "A": 3}
K = 60

rows = []
for d in ["A", "B", "C"]:
    r1, r2 = bm_rank.get(d), de_rank.get(d)
    s1 = 1 / (K + r1) if r1 else 0          # a doc missing from a list simply contributes 0 from that list
    s2 = 1 / (K + r2) if r2 else 0
    rows.append({"doc": d, "bm25_rank": r1, "dense_rank": r2, "from_bm25": round(s1, 5), "from_dense": round(s2, 5), "RRF": round(s1 + s2, 5)})

pd.DataFrame(rows).sort_values("RRF", ascending=False)

Read the table: `B` (consistently #2) edges out `C` (#1 in only one list). *Agreement between retrievers is rewarded.*

### Visual: how `k` changes RRF
The plot shows the weight of rank `r` relative to the weight of rank 1.

In [ ]:
ranks = np.arange(1, 21)
fig, ax = plt.subplots(figsize=(7, 3.8))
for k in [1, 10, 60, 200]:
    w = 1 / (k + ranks)
    ax.plot(ranks, w / w[0], marker="o", ms=3, label=f"k={k}")   # divide by rank-1 weight to compare shapes
ax.set_xlabel("rank in a list"); ax.set_ylabel("weight relative to rank 1")
ax.set_title("Small k: top ranks dominate. Large k: nearly flat")
ax.legend(); plt.show()

**Checkpoint 4.**
1. With `k=1`, how much more is rank 1 worth than rank 2? And with `k=200`?
2. Why is RRF robust to a retriever whose scores are "weird" (all zeros, huge outliers)?
3. When might a weighted blend beat RRF?

## 5. Evaluation: how do we know it works?

**Concept.** Never trust a retriever you have not measured. Three standard metrics, each answering a different question.
Let `k` be how many results you look at (here 3).

| Metric | Question it answers | Formula (intuition) |
|---|---|---|
| **Recall@k** | "Of all relevant docs, how many did I get in the top k?" | (relevant docs found in top-k) divided by (all relevant docs) |
| **MRR** | "How far down is the *first* relevant doc?" | `1 / rank of first relevant` (averaged over queries) |
| **nDCG@k** | "Are relevant docs near the *top*, not just present?" | discounted gain `1/log2(rank+1)`, divided by the best possible value |

Which to watch? For RAG, **Recall@k** matters most for the *first stage* (you cannot rerank what you did not retrieve).
**MRR / nDCG** matter more for the *final* ranking that goes to the LLM.

**Worked example.** Relevant = {1, 4}. Ranking = [1, 7, 4, ...], k=3.
- Recall@3 = 2/2 = 1.0 (both found).
- MRR = 1/1 = 1.0 (first relevant at rank 1).
- nDCG@3 = (1/log2(2) + 1/log2(4)) / (1/log2(2) + 1/log2(3)) = 1.5 / 1.631 = 0.92 (the second relevant doc is at rank 3, not 2, so a small penalty).

In [ ]:
def recall_at_k(ranking, relevant, k):
    # Set intersection of the top-k ids with the relevant ids, as a fraction of all relevant ids.
    return len(set(ranking[:k]) & relevant) / len(relevant)


def mrr(ranking, relevant):
    for r, d in enumerate(ranking, start=1):
        if d in relevant:
            return 1.0 / r            # reciprocal rank of the FIRST relevant hit
    return 0.0                        # never found: no credit


def ndcg_at_k(ranking, relevant, k):
    # DCG: sum of 1/log2(rank+1) over relevant docs in the top-k. Later ranks are discounted more.
    dcg = sum(1.0 / math.log2(r + 1) for r, d in enumerate(ranking[:k], start=1) if d in relevant)
    # Ideal DCG: what we'd get if all relevant docs were ranked first. Dividing by it scales the score to [0, 1].
    ideal = sum(1.0 / math.log2(r + 1) for r in range(1, min(len(relevant), k) + 1))
    return dcg / ideal


# Verify the worked example from the lecture notes.
print(round(ndcg_at_k([1, 7, 4, 0], {1, 4}, 3), 3), "(expected about 0.92)")


def evaluate(retrieve_fn, k=3):
    # Run one retriever over every labeled query and collect per-query metrics.
    rows = []
    for q, rel in QRELS.items():
        ranking = retrieve_fn(q)
        rows.append({"query": q, "recall": recall_at_k(ranking, rel, k),
                     "mrr": mrr(ranking, rel), "ndcg": ndcg_at_k(ranking, rel, k)})
    return pd.DataFrame(rows)


# A "system" is just a function: query -> ranked list of doc ids. Adding a new idea = adding a new entry.
SYSTEMS = {
    "bm25": lambda q: rank_from_scores(bm25_scores(q)),
    "dense": lambda q: rank_from_scores(dense_scores(q)),
    "hybrid_rrf": lambda q: hybrid_rrf(q),
    "weighted_0.5": lambda q: weighted(q, 0.5),
}

results = {name: evaluate(fn) for name, fn in SYSTEMS.items()}
# Average each metric over queries to get one summary row per system.
summary = pd.DataFrame({name: df[["recall", "mrr", "ndcg"]].mean() for name, df in results.items()}).T
summary.round(3)

In [ ]:
# Visual summary: grouped bars, one group per metric.
summary.plot.bar(figsize=(8, 3.8), rot=0, ylim=(0, 1.05))
plt.title("Average retrieval quality by system (k=3)")
plt.ylabel("score"); plt.legend(loc="lower right"); plt.show()

### What you should see, and a surprise
On this toy corpus **dense alone scores perfectly**, and plain RRF hybrid scores *below* it. Do not skip past this; it is the most
instructive result in the notebook.

**Why?** Look at what BM25 returns for a paraphrase query such as "I forgot my password". Only a couple of docs share any words.
Every other doc scores exactly **0**, and `argsort` still puts them in *some* order. RRF treats that arbitrary order as if it were
a real ranking and adds votes from it. BM25 is injecting **noise** into the fusion.

**The fix** is a principle worth remembering: *a retriever should only vote for documents it actually matched.*
Below we drop zero-score BM25 candidates before fusing.

**Honest conclusion:** with 18 easy documents, there is little for BM25 to add. Hybrid pays off when the corpus is large and full of
identifiers, near-duplicates and jargon (Exercises 6 and 8). Evaluating on *your* data is the only way to know.

In [ ]:
def hybrid_rrf_matched(query, k=60, pool=50):
    # Same as hybrid_rrf, but BM25 only votes for docs that matched at least one query term (score > 0).
    bm = bm25_scores(query)
    sparse = [DOC_IDS[i] for i in np.argsort(-bm) if bm[i] > 0][:pool]    # drop the zero-score filler
    dense = rank_from_scores(dense_scores(query))[:pool]
    return rrf([sparse, dense], k=k)


SYSTEMS["hybrid_rrf_matched"] = hybrid_rrf_matched
results["hybrid_rrf_matched"] = evaluate(hybrid_rrf_matched)
pd.DataFrame({n: df[["recall", "mrr", "ndcg"]].mean() for n, df in results.items()}).T.round(3)

**Professor's warning on small test sets.** With 14 queries, one query changes the average by 7 points.
Treat differences of a few points as noise. Real evaluation uses hundreds of queries and, ideally, confidence intervals.

## 6. Read the failures (the most valuable skill)

Averages hide the story. A system with the same mean as another may fail on *completely different* queries.
Always look at **per-query** results.

In [ ]:
# Per-query MRR for each system, as a heatmap. Dark = good (1.0), light = the first relevant doc was far down or missing.
per_query = pd.DataFrame({name: df.set_index("query")["mrr"] for name, df in results.items()})

fig, ax = plt.subplots(figsize=(8, 5.5))
im = ax.imshow(per_query.values, cmap="Blues", vmin=0, vmax=1, aspect="auto")
ax.set_xticks(range(per_query.shape[1])); ax.set_xticklabels(per_query.columns, rotation=20)
ax.set_yticks(range(per_query.shape[0])); ax.set_yticklabels(per_query.index, fontsize=9)
for i in range(per_query.shape[0]):
    for j in range(per_query.shape[1]):
        ax.text(j, i, f"{per_query.values[i, j]:.2f}", ha="center", va="center", fontsize=8,
                color="white" if per_query.values[i, j] > 0.6 else "black")
ax.set_title("Per-query MRR (rows = queries, columns = systems)")
plt.colorbar(im, ax=ax, fraction=0.04); plt.tight_layout(); plt.show()

In [ ]:
# Drill into ONE query across all systems. Change `q` to investigate others.
q = "how to get logs from a crashed pod"
print("Relevant doc ids:", QRELS[q])
for name, fn in SYSTEMS.items():
    print(f"--- {name}")
    show(q, fn(q))

**Checkpoint 5 (write your answers in the log at the bottom).**
1. Which queries does BM25 fail? What do they have in common? Which does dense fail?
2. Compare with your predictions in Checkpoint 1.
3. Does hybrid ever lose to the better single system? Why might that happen?
4. With only 18 documents, the corpus is too easy. What would change with 100,000 documents?

**A taxonomy of retrieval failures** (use it when debugging):
| Symptom | Likely cause | Fix |
|---|---|---|
| Exact ID not found | tokenizer split it; or dense-only | fix tokenizer; add BM25 |
| Paraphrase not found | vocabulary mismatch | add dense retrieval |
| Right doc retrieved but ranked 8th | weak first-stage ordering | add a reranker |
| Right info split across two chunks | chunking | change chunk size / overlap |
| Chunk lacks context ("it expires in 30 days") | chunk is ambiguous alone | Contextual Retrieval |

## 7. Reranking: precision on a short list

**Concept.** First-stage retrievers use a **bi-encoder**: the query and each document are encoded *separately*, so document
vectors can be precomputed and searched fast. The price: the model never sees the query and document *together*.

A **cross-encoder** feeds `[query, document]` into one transformer and outputs a relevance score. It captures fine-grained
interactions (negation, exact phrase relationships), so it is **much more accurate, and much slower** (one model call per candidate).

```
Bi-encoder:     encode(query)  .  encode(doc)      -> fast, precomputable, coarse
Cross-encoder:  model([query, doc])                -> slow, per-pair, precise
```

**The standard recipe:** use cheap methods to cut millions of chunks to ~50 to 200 candidates, then spend the expensive
model only on those. Latency stays bounded because the candidate count is bounded.

Even more advanced options (for later): **late interaction** models like ColBERT, and LLM-based rerankers.

In [ ]:
from sentence_transformers import CrossEncoder

# A small cross-encoder trained on the MS MARCO passage-ranking dataset.
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")


def hybrid_rerank(query, top_n=8):
    # Stage 1: cheap hybrid retrieval, keep only the top_n candidates.
    candidates = hybrid_rrf(query)[:top_n]
    # Stage 2: score every (query, doc) pair jointly. This is the expensive step, so top_n must stay small.
    scores = reranker.predict([(query, DOCS[d]) for d in candidates])
    # Re-order the candidates by the cross-encoder's scores, best first.
    return [candidates[i] for i in np.argsort(-scores)]


SYSTEMS["hybrid_rrf+rerank"] = hybrid_rerank
results["hybrid_rrf+rerank"] = evaluate(hybrid_rerank)
pd.DataFrame({n: df[["recall", "mrr", "ndcg"]].mean() for n, df in results.items()}).T.round(3)

**Checkpoint 6.** Did reranking help here? On a tiny corpus the first stage already returns nearly everything, so gains
are muted. Where would you expect the largest gains: small corpus, or a corpus of millions of chunks? Why?

## 8. Beyond the basics (reference)

### Contextual Retrieval (Anthropic)
Problem: chunks lose their surroundings. A chunk saying *"The company's revenue grew 3% over the previous quarter"* does not say
*which* company or quarter. Fix: before indexing, have an LLM write a 1 to 2 sentence **context** for each chunk
(situating it within its source document) and **prepend it** to the chunk text, for **both** the embeddings and the BM25 index.
Reported effect: about 49% fewer failed retrievals with contextual embeddings + contextual BM25, and about 67% when reranking is added.

### Other levers on your list
| Lever | What it changes |
|---|---|
| Chunking strategy | what the unit of retrieval *is* (often the biggest lever) |
| Query rewriting / expansion (HyDE, multi-query) | the query, to bridge vocabulary gaps |
| Metadata filters | restrict by date, source, permissions before or during retrieval |
| Learned sparse (SPLADE) | sparse vectors with *learned* term expansion, a middle path |
| Late interaction (ColBERT) | token-level matching, high quality, larger index |
| Domain-tuned embeddings | fine-tune the embedder on your data |

### Production notes
- **Vector databases** (Qdrant, Weaviate, Elasticsearch, Chroma, pgvector) do dense + sparse + fusion server-side; learn the concepts here first, then the API.
- **Latency budget:** retrieval stages run in parallel; the reranker usually dominates cost.
- **Freshness and updates:** two indexes means two things to keep in sync.

## 9. Exercises (do them in order)

After each, rerun the summary table and record what moved in the log at the bottom.

1. **RRF `k` sweep**: try `k` in 1, 10, 60, 200 (starter cell below).
2. **Alpha sweep**: for the weighted blend try alpha in 0, 0.25, 0.5, 0.75, 1. Confirm alpha=0 equals BM25-only and alpha=1 equals dense-only.
3. **BM25 params**: vary `k1` (0.5 to 2.0) and `b` (0 to 1). Use the plots in section 2 to predict the effect first.
4. **Tokenization**: add stemming (`nltk` PorterStemmer) so "timed" matches "time". What happens to `AX-9920` if you also split on hyphens?
5. **Embedding model**: swap in `BAAI/bge-small-en-v1.5`. Better or worse here? Why might it differ on a larger corpus?
6. **Harder data**: add 20 more documents with near-duplicates and distractors (for example error `E4012`). Does the gap between systems grow?
7. **Contextual Retrieval**: write a one-line context per doc (or have an LLM do it), prepend it, rebuild both indexes, compare.
8. **Scale up**: load a real dataset (for example BEIR SciFact via `ir_datasets`) and rerun the whole notebook. Which findings survive?

In [ ]:
# Starter for exercise 1: RRF k sweep.
# `k=k` in the lambda freezes the loop variable (a classic Python closure gotcha).
rows = []
for k in [1, 10, 60, 200]:
    df = evaluate(lambda q, k=k: hybrid_rrf(q, k=k))
    rows.append({"k": k, **df[["recall", "mrr", "ndcg"]].mean().round(3).to_dict()})
pd.DataFrame(rows)

In [ ]:
# Starter for exercise 2: alpha sweep for the weighted blend.
rows = []
for a in [0.0, 0.25, 0.5, 0.75, 1.0]:
    df = evaluate(lambda q, a=a: weighted(q, alpha=a))
    rows.append({"alpha": a, **df[["recall", "mrr", "ndcg"]].mean().round(3).to_dict()})
pd.DataFrame(rows)

## 10. Glossary

| Term | Meaning |
|---|---|
| **RAG** | Retrieval-Augmented Generation: fetch relevant text, give it to an LLM to answer from |
| **Chunk** | the unit of text that is indexed and retrieved |
| **Sparse retrieval** | vectors over the vocabulary, mostly zeros (BM25, SPLADE) |
| **Dense retrieval** | short, fully-populated embedding vectors from a neural model |
| **BM25** | probabilistic keyword ranking function with tf saturation and length normalization |
| **IDF** | inverse document frequency: rare terms weigh more |
| **Embedding** | a vector representing the meaning of text |
| **Cosine similarity** | angle-based similarity between vectors |
| **ANN / HNSW** | approximate nearest neighbor search / a graph-based ANN index |
| **RRF** | Reciprocal Rank Fusion: merge rankings using only ranks |
| **Bi-encoder** | encodes query and doc separately (fast) |
| **Cross-encoder** | scores query and doc jointly (accurate, slow) |
| **Recall@k** | fraction of relevant docs found in the top k |
| **MRR** | mean reciprocal rank of the first relevant doc |
| **nDCG** | rank-aware gain metric, scaled to [0, 1] |
| **qrels** | the labeled (query, relevant docs) test set |
| **Vocabulary mismatch** | query and document mean the same but share no words |

## 11. Cheat sheet

```
BM25   : sum_t IDF(t) * tf(k1+1) / (tf + k1(1 - b + b*dl/avgdl))      defaults k1~1.5, b=0.75
Cosine : u.v / (|u||v|)   (== dot product if normalized)
RRF    : sum_rankers 1/(k + rank)                                       default k=60
Blend  : alpha*dense_norm + (1-alpha)*bm25_norm                         tune alpha on eval set
Pipeline: retrieve(50-500 each) -> fuse -> rerank(top 50-200) -> LLM(top 3-10)
Debug  : eval set first, per-query failures second, tune knobs last
```

## 12. Reading list
**Overviews and tutorials**
- Hybrid Search for RAG (denser.ai): https://denser.ai/blog/hybrid-search-for-rag/
- Implementing hybrid semantic + lexical search (Machine Learning Mastery): https://machinelearningmastery.com/implementing-hybrid-semantic-lexical-search-in-rag/
- BM25, SPLADE and vector search (Prem AI): https://blog.premai.io/hybrid-search-for-rag-bm25-splade-and-vector-search-combined
- Hybrid search and reranking (WZ-IT): https://wz-it.com/en/knowledge/ki/hybrid-search-reranking/

**Contextual Retrieval**
- Anthropic: https://www.anthropic.com/research/contextual-retrieval
- InfoQ summary: https://www.infoq.com/news/2024/09/anthropic-contextual-retrieval/

**Engine documentation (when you move to production tooling)**
- Weaviate: https://docs.weaviate.io/weaviate/concepts/search/hybrid-search
- Qdrant: https://qdrant.tech/documentation/advanced-tutorials/reranking-hybrid-search/
- Chroma: https://docs.trychroma.com/cloud/search-api/hybrid-search

**Papers (search by title)**
- Cormack, Clarke, Buettcher (2009), "Reciprocal Rank Fusion outperforms Condorcet and individual rank learning methods"
- Robertson and Zaragoza (2009), "The Probabilistic Relevance Framework: BM25 and Beyond"
- Formal et al. (2021), "SPLADE: Sparse Lexical and Expansion Model for First Stage Ranking"
- Khattab and Zaharia (2020), "ColBERT: Efficient and Effective Passage Search via Contextualized Late Interaction"
- Thakur et al. (2021), "BEIR: A Heterogeneous Benchmark for Zero-shot Evaluation of Information Retrieval Models"

## 13. My experiments log

| # | Change | Recall@3 | MRR | nDCG@3 | What I learned |
|---|--------|----------|-----|--------|----------------|
| 0 | baseline hybrid_rrf (k=60) | | | | |
| 1 | | | | | |
| 2 | | | | | |

## Where next
1. Port this to **Qdrant** (dense + sparse vectors with server-side fusion).
2. Add an **LLM generation step** with citations, and evaluate *faithfulness* (does the answer follow from the sources?).
3. Replace the toy corpus with your own documents and build a real eval set from real user questions.